# NB3 — Core-7 × FashionCLIP embedding validation

Notebook này kiểm tra xem **mọi item còn lại sau Core-7/DROP** có embedding FashionCLIP dùng được hay không.

Nó kiểm tra:

1. cache đúng model `patrickjohncyh/fashion-clip`;
2. tensor có dạng `[N, 512]` và số hàng khớp `item_ids`;
3. không có duplicate ID, NaN/Inf, zero vector hoặc norm sai;
4. item trong từng `category_clean_{split}.jsonl` có metadata tương ứng;
5. mọi item cần dùng đều có embedding hợp lệ.

Nếu cả ba split pass, **không tạo thêm JSONL giống hệt**: các file `category_clean_*` hiện tại được dùng luôn làm final clean positives và pipeline có thể chuyển sang negative sampling.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy đúng code từ GitHub

Notebook mặc định dùng `main`. Khi test PR trước lúc merge, tạm đổi `REPO_REF` trong runtime thành tên feature branch; không cần lưu thay đổi đó vào notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
REPO_REF = 'main'  # Test trước merge: tạm đổi thành feature branch trong Colab.
REPO_DIR = Path('/content/opisoverated')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
print('Using repository ref:', REPO_REF)

## 3. Xác định input

- `CORE7_DIR`: chứa output full của NB2.
- `CACHE_PATH`: cache FashionCLIP mà leader đã tạo.

Cell tự thử hai tên folder Drive thường gặp để không phải sửa path nếu shortcut có tên `_shared`.

In [ ]:
CORE7_DIR = Path('/content/drive/MyDrive/fashion_audit/core7_drop_v1')

CACHE_CANDIDATES = [
    Path('/content/drive/MyDrive/PYML_Experimental_shared/fashionclip_item_embeddings.pt'),
    Path('/content/drive/MyDrive/PYML_Experimental/fashionclip_item_embeddings.pt'),
]
CACHE_PATH = next((path for path in CACHE_CANDIDATES if path.exists()), None)

if not CORE7_DIR.exists():
    raise FileNotFoundError(f'Không tìm thấy Core-7 output folder: {CORE7_DIR}')
if CACHE_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy fashionclip_item_embeddings.pt. '
        'Hãy thêm shortcut PYML_Experimental_shared vào My Drive hoặc sửa CACHE_CANDIDATES.'
    )

print('Core-7 folder:', CORE7_DIR)
print('Embedding cache:', CACHE_PATH)

## 4. Kiểm tra đủ 6 input JSONL

In [ ]:
SPLITS = ('train', 'valid', 'test')
POSITIVES = {
    split: CORE7_DIR / f'category_clean_{split}.jsonl'
    for split in SPLITS
}
METADATA = {
    split: CORE7_DIR / f'core7_item_metadata_v1_{split}.jsonl'
    for split in SPLITS
}
REPORT_PATH = CORE7_DIR / 'core7_embedding_validation_report.json'

for split in SPLITS:
    for kind, path in [('positive', POSITIVES[split]), ('metadata', METADATA[split])]:
        print(split, kind, path.name, 'exists=', path.exists())
        if not path.exists():
            raise FileNotFoundError(path)

## 5. Chạy cache audit + coverage join

Phần này chỉ đọc dữ liệu và ghi một report JSON nhỏ; không thay đổi ba dataset hiện tại.

In [ ]:
from src.data.validate_core7_embeddings import validate_core7_embedding_coverage

report = validate_core7_embedding_coverage(
    cache_path=CACHE_PATH,
    positives_by_split=POSITIVES,
    metadata_by_split=METADATA,
    report_path=REPORT_PATH,
)

print('Saved report:', REPORT_PATH)

## 6. Đọc kết quả ngắn gọn

In [ ]:
cache_report = report['cache']
print('CACHE')
print('  pass            :', cache_report['pass'])
print('  model           :', cache_report['model_id'])
print('  shape           :', (cache_report['embedding_row_count'], cache_report['embedding_dim']))
print('  usable items    :', cache_report['usable_item_count'])
print('  non-finite rows :', cache_report['nonfinite_row_count'])
print('  zero-norm rows  :', cache_report['zero_norm_row_count'])
print('  bad-norm rows   :', cache_report['bad_norm_row_count'])
print()

for split in SPLITS:
    split_report = report['splits'][split]
    print(split.upper())
    print('  pass                 :', split_report['pass'])
    print('  positive outfits     :', split_report['positive_sample_count'])
    print('  required unique items:', split_report['unique_required_item_count'])
    print('  embedding coverage   :', f"{split_report['embedding_coverage']:.4%}")
    print('  missing/invalid      :', split_report['missing_or_invalid_embedding_count'])
    print()

print('OVERALL PASS                 :', report['pass'])
print('REUSE CATEGORY-CLEAN AS FINAL:', report['reuse_category_clean_as_final'])
print('READY FOR NEGATIVE SAMPLING  :', report['ready_for_negative_sampling'])

## 7. Cách hiểu output

Nếu ba dòng cuối đều là `True`:

- không tạo thêm `final_clean_*.jsonl`;
- dùng chính `category_clean_train/valid/test.jsonl` làm final clean positives;
- file `core7_embedding_validation_report.json` là bằng chứng validation;
- bước tiếp theo là viết negative sampler mới.

Nếu `False`, xem các field `*_examples` trong report. Core code đã có hàm `repair_split`, nhưng **không tự sửa âm thầm** trước khi nhóm đọc nguyên nhân lỗi.